# Phase C1: Structured Profiles Acceptance

This notebook demonstrates the offline structured profile API without touching hardware.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path().resolve().parent / 'src'))

from awr2944_dca.api.profile import RadarProfile
from awr2944_dca.lab import RadarProject
import json

## 1. Load smoke_v1 and Summary

In [ ]:
smoke = RadarProfile.smoke_v1()
print(f"Profile: {smoke.name}")
print(f"Description: {smoke.description}")

## 2. Validation Report & Derived Values

In [ ]:
report = smoke.validate()
print(report.summary())
print("\nDerived Values:")
for k, v in report.derived.items():
    print(f"  {k}: {v}")

plan = smoke.byte_plan(guard_frames=2)
print("\nByte Plan (with 2 guard frames):")
for k, v in plan.items():
    print(f"  {k}: {v}")

## 3. to_dsp_profile Exact Mapping

In [ ]:
dsp = smoke.to_dsp_profile()
print(f"Start Freq (Hz): {dsp.start_frequency_hz}")
print(f"Slope (Hz/s): {dsp.slope_hz_per_s}")
print(f"ADC Sample Rate (Hz): {dsp.adc_sample_rate_hz}")
print(f"ADC Samples: {dsp.adc_samples}")
print(f"Idle Time (s): {dsp.idle_time_s}")
print(f"Ramp End Time (s): {dsp.ramp_end_time_s}")
print(f"Chirps/Frame: {dsp.chirps_per_frame}")
print(f"Frame Count: {dsp.frame_count}")
print(f"Frame Period (s): {dsp.frame_period_s}")
print(f"RX Count: {dsp.rx_count}")
print(f"TX Mask: {dsp.tx_mask}")
print(f"Sample Format: {dsp.sample_format}")
print(f"Cube Layout: {dsp.cube_layout}")

## 4. Unchanged-builder Command Export (Golden Parity)

In [ ]:
cmds = smoke.to_sdk_cli()
for cmd in cmds:
    print(cmd)
    
cmd_text = " ".join(cmds)
assert "sensorStart" not in cmd_text, "sensorStart found!"
assert "sensorStop" not in cmd_text, "sensorStop found!"
print("\nVerified: sensorStart and sensorStop are absent.")

## 5. TOML Round Trip & Phase A Minimal-Profile Compatibility

In [ ]:
toml_str = smoke.to_toml()
recovered = RadarProfile.from_toml(toml_str)
assert smoke == recovered
print("TOML round-trip successful.")

# Minimal profile compat
minimal_toml = """\
schema_version = "1.0"
name = "phase_a_minimal"
"""
minimal = RadarProfile.from_toml(minimal_toml)
print(f"Minimal profile loaded. Name: {minimal.name}, Start Freq: {minimal.rf.start_frequency_ghz}")

## 6. Supported Frame-Count/Chirp-Count Variant

In [ ]:
variant = smoke.with_frame(frame_count=16, chirps_per_frame=64)
cmds = variant.to_sdk_cli()
print("Supported variant compiled successfully.")
print("Frame config command:")
print([c for c in cmds if c.startswith("frameCfg")][0])

## 7. Unsupported Frame-Period and RF Variant Rejection

In [ ]:
from awr2944_dca.api.profile import ProfileCompilationNotSupported

try:
    bad_frame = smoke.with_frame(frame_period_ms=20.0)
    bad_frame.to_sdk_cli()
    print("FAIL: Should have rejected frame period change.")
except ProfileCompilationNotSupported as e:
    print("Correctly rejected frame period change:")
    print(e)

try:
    bad_rf = smoke.with_rf(slope_mhz_per_us=35.0)
    bad_rf.to_sdk_cli()
    print("FAIL: Should have rejected RF change.")
except ProfileCompilationNotSupported as e:
    print("Correctly rejected RF change:")
    print(e)

## 8. Collection Origin Reporting, Save/Load/Delete & Overwrite Protection

In [ ]:
# Create temporary project
import tempfile
with tempfile.TemporaryDirectory() as td:
    proj = RadarProject.create("TempProj", td)
    
    print("Initial profiles:")
    for p in proj.profiles.list():
        print(f"  {p.name} (Origin: {p.origin})")
        
    # Save a custom profile
    custom = smoke.rename("custom_profile")
    proj.profiles.save(custom)
    
    print("\nAfter saving custom_profile:")
    for p in proj.profiles.list():
        print(f"  {p.name} (Origin: {p.origin})")
        
    # Overwrite protection
    try:
        proj.profiles.save(custom)
        print("FAIL: Should have prevented overwrite without explicit flag.")
    except FileExistsError as e:
        print(f"\nOverwrite prevented: {e}")
        
    proj.profiles.save(custom, overwrite=True)
    print("Overwrite with flag=True succeeded.")
    
    # Delete project profile
    proj.profiles.delete("custom_profile")
    print("\nAfter deleting custom_profile:")
    for p in proj.profiles.list():
        print(f"  {p.name} (Origin: {p.origin})")
        
    # Delete builtin prevention
    try:
        proj.profiles.delete("smoke_v1")
    except ValueError as e:
        print(f"\nBuilt-in deletion prevented: {e}")

## 9. Final Marker

In [ ]:
print("PHASE_C1_STRUCTURED_PROFILES_PASS")